# Galassi 2012 - Gate D: Cold-Fill Comparator (Optional)

This notebook implements the optional Gate D. It compares a reference fill (H2_250 conditions) against a "Cold Fill" scenario (270 K).

**Requirements:**
- Gate C must have passed (requires `locked_params.json`).

In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add repo to path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/H2_Storage'
except ImportError:
    BASE_DIR = 'H2_Storage'
    print(f"Using local BASE_DIR: {BASE_DIR}")

REPO_DIR = os.path.join(BASE_DIR, 'repo')
sys.path.append(REPO_DIR)

try:
    from h2tank.galassi_baseline import simulate_fast_fill
except ImportError:
    # Attempt relative import
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../repo')))
    from h2tank.galassi_baseline import simulate_fast_fill

DATA_DIR = os.path.join(BASE_DIR, 'validation/data/galassi_2012')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

## Load Requirements

In [ ]:
locked_params_path = os.path.join(RESULTS_DIR, 'galassi2012_locked_params.json')
if not os.path.exists(locked_params_path):
    print("STOP: results/galassi2012_locked_params.json is missing. Run Gate C first.")
    raise FileNotFoundError("Locked parameters missing.")

with open(locked_params_path, 'r') as f:
    locked_params = json.load(f)
    
print("Loaded Locked Parameters:", locked_params)

protocol_path = os.path.join(DATA_DIR, 'protocol_table1.json')
with open(protocol_path, 'r') as f:
    protocols = json.load(f)
    
base_protocol = protocols['H2_250']
print("Loaded H2_250 Protocol:", base_protocol)

## Setup Simulations

In [ ]:
# Reference Case
# Tini=21C, Tamb=18C (Matches H2_250)
# Inflow Temperature assumption: usually Tamb if not specified. 
# The baseline model defaults T_inflow to Tamb.
ref_protocol = base_protocol.copy()
ref_protocol['Tini_C'] = 21
ref_protocol['Tamb_C'] = 18

ref_params = locked_params.copy()
# Explicitly set T_inflow to Tamb (18C = 291.15K) for clarity
ref_params['T_inflow_K'] = 18 + 273.15


# Cold Fill Case
# "set initial gas temperature to 270 K"
# Assumption: Inflow is also 270 K.
cold_protocol = base_protocol.copy()
cold_protocol['Tini_C'] = 270 - 273.15 # -3.15 C
cold_protocol['Tamb_C'] = 18 # Ambient is likely still 18C?

cold_params = locked_params.copy()
cold_params['T_inflow_K'] = 270.0 # Cold inflow

print("Reference T_ini: 21 C, T_inflow: 18 C")
print(f"Cold Fill T_ini: {cold_protocol['Tini_C']:.2f} C, T_inflow: 270 K")

## Run Simulations

In [ ]:
print("Running Reference Simulation...")
res_ref = simulate_fast_fill(ref_protocol, ref_params)
print(f"  Reference Peak T: {res_ref['T_peak_K']:.2f} K")

print("Running Cold Fill Simulation...")
res_cold = simulate_fast_fill(cold_protocol, cold_params)
print(f"  Cold Fill Peak T: {res_cold['T_peak_K']:.2f} K")

## Plotting

In [ ]:
plt.figure(figsize=(10, 6))

# Convert K to C for plot (optional, but 85C limit is in C usually, prompt says 85C)
# Prompt says "Include 85C limit line".
t_ref = res_ref['time_s']
T_ref_C = res_ref['T_K'] - 273.15

t_cold = res_cold['time_s']
T_cold_C = res_cold['T_K'] - 273.15

plt.plot(t_ref, T_ref_C, label='Reference (Inflow ~18°C)')
plt.plot(t_cold, T_cold_C, label='Cold Fill (Inflow 270 K)')

# 85C Limit
plt.axhline(y=85, color='r', linestyle='--', label='85°C Limit')

plt.xlabel('Time (s)')
plt.ylabel('Temperature (°C)')
plt.title('Galassi 2012: Cold Fill Comparator (Gate D)')
plt.legend()
plt.grid(True)

fig_path = os.path.join(FIGURES_DIR, 'galassi2012_coldfill_comparator.png')
plt.savefig(fig_path)
print(f"Saved plot to {fig_path}")
plt.close()

## Metrics and Checkpoint

In [ ]:
# Metrics
T_peak_ref_K = res_ref['T_peak_K']
T_peak_cold_K = res_cold['T_peak_K']

delta_T_peak_reduction = T_peak_ref_K - T_peak_cold_K
peak_cold_C = T_peak_cold_K - 273.15
is_under_85C = peak_cold_C < 85.0

metrics_data = [{
    'Reference_Peak_K': T_peak_ref_K,
    'Cold_Peak_K': T_peak_cold_K,
    'Delta_T_Peak_Reduction_K': delta_T_peak_reduction,
    'Cold_Peak_C': peak_cold_C,
    'Is_Under_85C': is_under_85C
}]

df_metrics = pd.DataFrame(metrics_data)
csv_path = os.path.join(RESULTS_DIR, 'galassi2012_coldfill_metrics.csv')
df_metrics.to_csv(csv_path, index=False)
print(f"Saved metrics to {csv_path}")

# Checkpoint
checkpoint_d_data = {
    'ref_protocol': ref_protocol,
    'cold_protocol': cold_protocol,
    'ref_results': res_ref,
    'cold_results': res_cold,
    'metrics': metrics_data
}

checkpoint_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateD.pkl')
with open(checkpoint_path, 'wb') as f:
    pickle.dump(checkpoint_d_data, f)
print(f"Saved checkpoint to {checkpoint_path}")